In [41]:
#IMPORTING THE NECESSARY LIBRARIES

In [42]:
!pip install catboost

In [43]:
!pip install hvplot

In [44]:
!pip install xgboost

In [45]:
!pip install lightgbm

In [46]:
# Installing missing packages
!pip install xgboost lightgbm catboost

# Loading all Packages
print("==================== BLOCK 1 Started! ======================")

# Library for Data Manipulation
import numpy as np
import pandas as pd

# Library for Data Visualization.
import seaborn as sns
import matplotlib.pyplot as plt
import hvplot

%matplotlib inline
sns.set_style("whitegrid")
plt.style.use("fivethirtyeight")

# Library for Statistical Modelling
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score, precision_recall_curve, roc_curve
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import AdaBoostClassifier

print("==================== Packages Loaded ======================")

# Library to ignore the warnings
import warnings
warnings.filterwarnings('always')
warnings.filterwarnings('ignore')

print("==================== BLOCK 1 Completed! ======================")


==================== BLOCK 1 Started! ======================
==================== Packages Loaded ======================
==================== BLOCK 1 Completed! ======================


In [47]:
#LOADING THE DATASET

In [48]:
employee_data = pd.read_csv(r"C:\Users\sinch\Downloads\IBM-HR-Analytics-Employee-Attrition-and-Performance-Revised.csv")

In [49]:
# Print top 5 rows in the dataframe.
employee_data.head().style.set_properties(**{'background-color': '#E9F6E2','color': 'black','border-color': '#8b8c8c'})

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,College,Life Sciences,Medium,Female,94,High,Junior Level,Sales Executive,Very High,Single,5993,19479,8,Yes,11,Excellent,Low,0,8,0,Bad,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,Below College,Life Sciences,High,Male,61,Medium,Junior Level,Research Scientist,Medium,Married,5130,24907,1,No,23,Outstanding,Very High,1,10,3,Better,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,College,Other,Very High,Male,92,Medium,Entry Level,Laboratory Technician,High,Single,2090,2396,6,Yes,15,Excellent,Medium,0,7,3,Better,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,Master,Life Sciences,Very High,Female,56,High,Entry Level,Research Scientist,High,Married,2909,23159,1,Yes,11,Excellent,High,0,8,3,Better,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,Below College,Medical,Low,Male,40,High,Entry Level,Laboratory Technician,Medium,Married,3468,16632,9,No,12,Excellent,Very High,1,6,3,Better,2,2,2,2


In [50]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

# Load the dataset
employee_data = pd.read_csv(r"C:\Users\sinch\Downloads\IBM-HR-Analytics-Employee-Attrition-and-Performance-Revised.csv")

# Convert object columns to categorical
for col in employee_data.select_dtypes(include=['object']).columns:
    employee_data[col] = employee_data[col].astype('category')

# Define categorical and numerical columns
categorical_cols = employee_data.select_dtypes(include=['category']).columns
numerical_cols = employee_data.select_dtypes(include=['int64', 'float64']).columns

# Encoding categorical data
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    employee_data[col] = le.fit_transform(employee_data[col])
    label_encoders[col] = le

# Scaling numerical data
scaler = StandardScaler()
employee_data[numerical_cols] = scaler.fit_transform(employee_data[numerical_cols])

# Splitting into training and testing sets
X = employee_data.drop(columns=['Attrition'])
y = employee_data['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize models
logreg_model = LogisticRegression(random_state=42)
rf_model = RandomForestClassifier(random_state=42)
xgb_model = XGBClassifier(random_state=42)

# Fit models on training data
logreg_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

# Create ensemble model
ensemble_model = VotingClassifier(estimators=[
    ('logreg', logreg_model),
    ('rf', rf_model),
    ('xgb', xgb_model)
], voting='soft')  # 'soft' voting for probability averaging

# Fit ensemble model
ensemble_model.fit(X_train, y_train)

# Function to print evaluation metrics
def evaluate_model(model_name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    print(f"Evaluation Metrics for {model_name}:")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))

# Evaluate individual models
evaluate_model("Logistic Regression", logreg_model, X_test, y_test)
evaluate_model("Random Forest", rf_model, X_test, y_test)
evaluate_model("XGBoost", xgb_model, X_test, y_test)

# Evaluate ensemble model
evaluate_model("Ensemble Model", ensemble_model, X_test, y_test)

# Pickle the models and artifacts
pickle.dump(logreg_model, open('logreg_model.pkl', 'wb'))
pickle.dump(rf_model, open('rf_model.pkl', 'wb'))
pickle.dump(xgb_model, open('xgb_model.pkl', 'wb'))
pickle.dump(ensemble_model, open('ensemble_model.pkl', 'wb'))
pickle.dump(label_encoders, open('label_encoders.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))
pickle.dump(numerical_cols, open('numerical_cols.pkl', 'wb'))
pickle.dump(categorical_cols, open('categorical_cols.pkl', 'wb'))

print("Models and artifacts have been pickled successfully.")


Evaluation Metrics for Logistic Regression:
Accuracy: 0.8639455782312925
Confusion Matrix:
 [[247   8]
 [ 32   7]]
Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.97      0.93       255
           1       0.47      0.18      0.26        39

    accuracy                           0.86       294
   macro avg       0.68      0.57      0.59       294
weighted avg       0.83      0.86      0.84       294

Evaluation Metrics for Random Forest:
Accuracy: 0.8809523809523809
Confusion Matrix:
 [[254   1]
 [ 34   5]]
Classification Report:
               precision    recall  f1-score   support

           0       0.88      1.00      0.94       255
           1       0.83      0.13      0.22        39

    accuracy                           0.88       294
   macro avg       0.86      0.56      0.58       294
weighted avg       0.88      0.88      0.84       294

Evaluation Metrics for XGBoost:
Accuracy: 0.8639455782312925
Confusion Mat

In [51]:
import pickle

# Assuming ensemble_model is your trained model
pickle.dump(ensemble_model, open('model.pkl', 'wb'))


In [52]:
with open('label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)